# Frequentist coverage of the 95% credible intervals (antithetic integral controller)

Figure 1 and Table 2 of the manuscript use prefixes of **one** trajectory, so their 24 intervals are not independent
checks of coverage. Here we simulate `N_REP` independent trajectories of the antithetic integral controller up to
`T_COV`, run the spike-and-slab sampler on each (same prior, sampler and settings as every example in the paper), and
count how often each 95% credible interval contains the true rate.

Also reported, per replicate:
* whether the median probability model ($\Pr(\text{off})<1/2$) equals the true network in every observed channel;
* for the channel $(+1,0,0)$, where $\varnothing\to X$ and $Z_1\to Z_1+X$ are both active, whether the true pair lies in
  the joint 95% credible region (normal approximation to the posterior of the two rates: squared Mahalanobis distance
  below the $\chi^2_2$ 95% quantile).

Each replicate is saved to `results/coverage_study/rep_XXX.csv` as soon as it finishes, so an interrupted run resumes
where it stopped. `RUN = False` only reads the saved replicates. A full run takes about 2 core-minutes per replicate.
Writes `writing/table_coverage.tex`.

In [1]:
import os, sys, io, contextlib, glob
import numpy as np
import pandas as pd
from scipy.stats import chi2, beta as beta_dist
from joblib import Parallel, delayed

sys.path.append(os.path.abspath(".."))
from src.parsing import load_reaction_network, generate_single_trajectory, generate_reactions, parse_trajectory
from src.inference import extract_local_data
from src.mcmc import adaptive_mcmc_spike_slab

RUN = False          # True: simulate and run the sampler for the replicates that are not saved yet
N_REP = 100          # independent trajectories
T_COV = 500          # observation time of each trajectory
N_JOBS = 2           # parallel workers
SEED = 7             # trajectory r uses seed 7000 + r; the sampler of channel i uses streams (SEED, r, i, 0/1)

# MCMC settings: identical to every example in the manuscript (src/mcmc.py, run_channel_mcmc)
NIterates, Burnin, Thinout = 500_000, 50_000, 100
ALPHA, BETA, PI = 2, 0.25, 0.75
EPS_OFF = 1e-3       # a draw counts as "off" if the rate is below this (as in summarize_chains)

NETWORK_FILE = "../data/example6_network.json"
OUT_DIR = "../results/coverage_study"
TEX_OUT = "../writing/table_coverage.tex"
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
(reactionNetwork, _, _, _, trueTheta, parameter_values, _,
 reactant_matrix, unique_changes, compatible_reactions, species_names) = load_reaction_network(NETWORK_FILE)

complexes = np.array([
    [0, 1, 0, 0, 2, 0, 0, 1, 1, 0],   # X
    [0, 0, 1, 0, 0, 2, 0, 1, 0, 1],   # Z1
    [0, 0, 0, 1, 0, 0, 2, 0, 1, 1],   # Z2
])
_, _, _, reaction_names, _, _, _ = generate_reactions(complexes, species_names)
reaction_names = [r.rstrip(":") for r in reaction_names]
ACTIVE = [i for i in range(len(trueTheta)) if trueTheta[i] > 0]
print({reaction_names[i]: trueTheta[i] for i in ACTIVE})

Reaction network loaded from ../data/example6_network.json
  Species:   ['X', 'Z1', 'Z2']
  Reactions: 6 sampled  |  90 total in full CRN
  Unique stoichiometric changes: 54
{'Empty_to_X': np.float64(2.0), 'Empty_to_Z1': np.float64(6.0), 'X_to_Empty': np.float64(1.0), 'X_to_X+Z2': np.float64(2.0), 'Z1_to_X+Z1': np.float64(1.0), 'Z1+Z2_to_Empty': np.float64(1.0)}


In [3]:
def run_replicate(r):
    """Simulate trajectory r, run the sampler on every observed channel, save one row per candidate reaction."""
    path = os.path.join(OUT_DIR, f"rep_{r:03d}.csv")
    if os.path.exists(path):
        return path
    with contextlib.redirect_stdout(io.StringIO()):
        times, states = generate_single_trajectory(reactionNetwork, parameter_values, species_names,
                                                   finalTime=T_COV, minVal=[0, 0, 0], maxVal=[0, 0, 0],
                                                   seed=1000 * SEED + r)
        _, jump_counts, waiting_times, propensities = parse_trajectory(
            states, times, reactant_matrix, unique_changes, compatible_reactions, verbose=False)
    rows = []
    for index, deltaX in enumerate(unique_changes):
        with contextlib.redirect_stdout(io.StringIO()):
            local_counts, local_wt, local_prop, dx = extract_local_data(
                jump_counts, waiting_times, propensities, unique_changes, index=index, deltaX=deltaX, verbose=False)
        n = sum(local_counts.values())
        if n == 0:
            continue
        cands = compatible_reactions[dx]
        theta_init = np.random.default_rng([SEED, r, index, 0]).uniform(0.001, 2.0, size=len(cands))
        chain = adaptive_mcmc_spike_slab(local_counts, local_wt, local_prop, theta_init, trueTheta,
                                         num_iterations=NIterates, alpha=ALPHA, beta=BETA, pi=PI,
                                         burn_in=Burnin, adapt_every_n=10, printEveryNSteps=0,
                                         seed=[SEED, r, index, 1])
        draws = chain[Burnin::Thinout]
        truth = np.array([trueTheta[c] for c in cands])
        on = truth > 0
        joint_ok = np.nan
        if on.sum() >= 2:                               # joint region of the active rates of the channel
            x = draws[:, on]; m = x.mean(0); d = truth[on] - m
            joint_ok = float(d @ np.linalg.solve(np.cov(x.T), d) <= chi2.ppf(0.95, on.sum()))
        poff = np.mean(np.abs(draws) < EPS_OFF, axis=0)
        lo, hi = np.quantile(draws, [0.025, 0.975], axis=0)
        for j, c in enumerate(cands):
            rows.append({"rep": r, "channel": str(tuple(int(v) for v in dx)), "events": n,
                         "reaction": reaction_names[c], "true": truth[j], "mean": draws[:, j].mean(),
                         "lo": lo[j], "hi": hi[j], "Pr_off": poff[j], "joint_ok": joint_ok})
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

if RUN:
    todo = [r for r in range(N_REP) if not os.path.exists(os.path.join(OUT_DIR, f"rep_{r:03d}.csv"))]
    print(f"{len(todo)} replicates to run")
    Parallel(n_jobs=N_JOBS, verbose=5)(delayed(run_replicate)(r) for r in todo)

In [4]:
files = sorted(glob.glob(os.path.join(OUT_DIR, "rep_*.csv")))
res = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
n_rep = res["rep"].nunique()
print(f"{n_rep} replicates, T = {T_COV}")

def cp_interval(k, n):            # Clopper-Pearson 95% interval for a binomial proportion
    lo = beta_dist.ppf(0.025, k, n - k + 1) if k > 0 else 0.0
    hi = beta_dist.ppf(0.975, k + 1, n - k) if k < n else 1.0
    return lo, hi

act = res[res["true"] > 0].copy()
act["covers"] = (act["lo"] <= act["true"]) & (act["true"] <= act["hi"])
act["rel_width"] = (act["hi"] - act["lo"]) / act["true"]
act["rel_bias"] = act["mean"] / act["true"] - 1
summary = []
for name in [reaction_names[i] for i in ACTIVE]:
    g = act[act["reaction"] == name]
    k, n = int(g["covers"].sum()), len(g)
    summary.append({"reaction": name, "true": g["true"].iloc[0], "n": n, "covered": k, "coverage": k / n,
                    "cp_lo": cp_interval(k, n)[0], "cp_hi": cp_interval(k, n)[1],
                    "missed_low": int((g["hi"] < g["true"]).sum()), "missed_high": int((g["lo"] > g["true"]).sum()),
                    "mean_rel_bias": g["rel_bias"].mean(), "mean_rel_width": g["rel_width"].mean()})
summary = pd.DataFrame(summary)
k, n = int(act["covers"].sum()), len(act)
print(summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nall active rates: {k}/{n} = {k/n:.3f}, 95% CI [{cp_interval(k, n)[0]:.3f}, {cp_interval(k, n)[1]:.3f}]")

pair = res[(res["channel"] == "(1, 0, 0)") & res["joint_ok"].notna()].drop_duplicates("rep")
kj = int(pair["joint_ok"].sum()); nj = len(pair)
print(f"joint 95% region of (Empty->X, Z1->X+Z1) contains the truth: {kj}/{nj}")

res["included"] = res["Pr_off"] < 0.5
res["correct"] = res["included"] == (res["true"] > 0)
struct = res.groupby("rep")["correct"].all()
inactive = res[res["true"] == 0]
print(f"median probability model = true network (all channels): {int(struct.sum())}/{n_rep}")
print(f"inactive candidates: smallest Pr(off) {inactive['Pr_off'].min():.3f}; "
      f"active reactions: largest Pr(off) {act['Pr_off'].max():.3f}")

100 replicates, T = 500
      reaction  true   n  covered  coverage  cp_lo  cp_hi  missed_low  missed_high  mean_rel_bias  mean_rel_width
    Empty_to_X 2.000 100       94     0.940  0.874  0.978           4            2         -0.003           0.160
   Empty_to_Z1 6.000 100       91     0.910  0.836  0.958           4            5          0.002           0.073
    X_to_Empty 1.000 100       96     0.960  0.901  0.989           2            2          0.003           0.101
     X_to_X+Z2 2.000 100       93     0.930  0.861  0.971           4            3         -0.002           0.071
    Z1_to_X+Z1 1.000 100       94     0.940  0.874  0.978           3            3         -0.001           0.258
Z1+Z2_to_Empty 1.000 100       94     0.940  0.874  0.978           1            5         -0.001           0.071

all active rates: 562/600 = 0.937, 95% CI [0.914, 0.955]
joint 95% region of (Empty->X, Z1->X+Z1) contains the truth: 95/100
median probability model = true network (all channel

In [5]:
def tex_rxn(name):
    lhs, rhs = name.split("_to_")
    conv = lambda s: r"\varnothing" if s == "Empty" else s.replace("Z1", "Z_1").replace("Z2", "Z_2")
    return f"${conv(lhs)} \\to {conv(rhs)}$"

L = [r"% Coverage of the 95% credible intervals over independent trajectories (antithetic integral controller)",
     r"% Generated by examples/coverage_study.ipynb -- do not edit by hand",
     r"\begin{table}[htbp]", r"\centering", r"\footnotesize",
     r"\caption{Antithetic integral controller: frequentist coverage of the 95\% credible intervals over "
     + f"{n_rep}" + r" independent trajectories observed up to $T=" + f"{T_COV}" + r"$, with the prior and sampler "
     r"of \Cref{sec:aic}. Coverage: fraction of trajectories whose interval contains the true rate $k^*$, with a "
     r"95\% Clopper--Pearson interval; misses below and above the interval; mean of $\hat k/k^*-1$ and of the "
     r"interval width divided by $k^*$. The interval for all six rates together is approximate, since rates "
     r"estimated from the same trajectory are not independent. The joint 95\% credible region of the two rates of channel "
     r"$+1X$ contains the true pair in " + f"{kj}" + r" of " + f"{nj}" + r" trajectories, and the median "
     r"probability model equals the true network in " + f"{int(struct.sum())}" + r" of " + f"{n_rep}" + r".}",
     r"\label{tab:coverage}",
     r"\begin{tabular}{l r r l c r r}", r"\toprule",
     r"Reaction & $k^*$ & Coverage & 95\% CI & Misses (low/high) & Bias & Width \\", r"\midrule"]
for _, s in summary.iterrows():
    L.append(f"{tex_rxn(s['reaction'])} & {s['true']:g} & {s['coverage']:.2f} & [{s['cp_lo']:.2f},\\,{s['cp_hi']:.2f}] & "
             f"{s['missed_low']}/{s['missed_high']} & ${s['mean_rel_bias']:+.3f}$ & {s['mean_rel_width']:.3f}" + r" \\")
L.append(r"\midrule")
lo_all, hi_all = cp_interval(k, n)
L.append(r"All six & & " + f"{k/n:.2f} & [{lo_all:.2f},\\,{hi_all:.2f}] & & & " + r"\\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
open(TEX_OUT, "w").write("\n".join(L) + "\n")
print("\n".join(L))

% Coverage of the 95% credible intervals over independent trajectories (antithetic integral controller)
% Generated by examples/coverage_study.ipynb -- do not edit by hand
\begin{table}[htbp]
\centering
\footnotesize
\caption{Antithetic integral controller: frequentist coverage of the 95\% credible intervals over 100 independent trajectories observed up to $T=500$, with the prior and sampler of \Cref{sec:aic}. Coverage: fraction of trajectories whose interval contains the true rate $k^*$, with a 95\% Clopper--Pearson interval; misses below and above the interval; mean of $\hat k/k^*-1$ and of the interval width divided by $k^*$. The interval for all six rates together is approximate, since rates estimated from the same trajectory are not independent. The joint 95\% credible region of the two rates of channel $+1X$ contains the true pair in 95 of 100 trajectories, and the median probability model equals the true network in 100 of 100.}
\label{tab:coverage}
\begin{tabular}{l r r l c r r}